# ASSOCIATION RULES

In [1]:
## • The Objective of this assignment is to introduce students to rule mining techniques, particularly focusing on market basket analysis and provide hands on experience.

### Dataset:

In [2]:
## • Use the Online retail dataset to apply the association rules.

### Data Preprocessing:

In [3]:
## • Pre-process the dataset to ensure it is suitable for Association rules, this may include handling missing values, removing duplicates, and converting the data to appropriate format.  

In [4]:
## Import required libraries..
import pandas as pd
import numpy as np

In [5]:
df = pd.read_excel("Online_retail.xlsx")
df.head()

,"shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil"
0,"burgers,meatballs,eggs"
1,chutney
2,"turkey,avocado"
3,"mineral water,milk,energy bar,whole wheat rice..."
4,low fat yogurt


In [6]:
## The data is already in transaction format but cannot be used directly for apriori..
## It needs to be converted to one-hot encoded basket format..

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                           Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                           --------------  ----- 
 0   shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil  7500 non-null   object
dtypes: object(1)
memory usage: 58.7+ KB


In [8]:
df.describe()

,"shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil"
count,7500
unique,5175
top,cookies
freq,223


In [9]:
## Split items into list because right now my data is in single string format..
## Apriori takes data in list of strings format..
transactions = df.iloc[:, 0].apply(lambda x: x.split(','))

In [10]:
transactions.head()

0                           [burgers, meatballs, eggs]
1                                            [chutney]
2                                    [turkey, avocado]
3    [mineral water, milk, energy bar, whole wheat ...
4                                     [low fat yogurt]
Name: shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil, dtype: object

In [11]:
## There could be unneccessary extra spaces in the data to avoid duplicate columns.. we need to remove them and clean item names..
transactions = transactions.apply(lambda x: [i.strip() for i in x])

In [12]:
transactions.head()

0                           [burgers, meatballs, eggs]
1                                            [chutney]
2                                    [turkey, avocado]
3    [mineral water, milk, energy bar, whole wheat ...
4                                     [low fat yogurt]
Name: shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil, dtype: object

In [13]:
## !pip install mlxtend
## Use the above code incase you are using mlxtend for the 1st time.. 

In [14]:
## one-hot encoding..
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns = te.columns_)

In [15]:
df_encoded.head()

,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,True,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [16]:
print(df_encoded.columns)

Index(['almonds', 'antioxydant juice', 'asparagus', 'avocado', 'babies food',
       'bacon', 'barbecue sauce', 'black tea', 'blueberries', 'body spray',
       ...
       'turkey', 'vegetables mix', 'water spray', 'white wine',
       'whole weat flour', 'whole wheat pasta', 'whole wheat rice', 'yams',
       'yogurt cake', 'zucchini'],
      dtype='object', length=119)


In [17]:
## lets convert True/False in the form of 0/1.. because apriori takes numeric..
df_encoded = df_encoded.astype(int)
df_encoded.head()

,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Association Rule Mining:

In [18]:
## • Implement an Apriori algorithm using tool like python with libraries such as Pandas and Mlxtend etc.
## • Apply association rule mining techniques to the pre-processed dataset to discover interesting relationships between products purchased together.
## • Set appropriate threshold for support, confidence and lift to extract meaning full rules.

In [19]:
## lets apply apriori algorithm and find frequent itemsets..
from mlxtend.frequent_patterns import apriori
frequent_itemsets = apriori(df_encoded, min_support = 0.02, use_colnames = True)

## min_support = 0.02 means items must appears in at least 2% transactions..

C:\Users\devlok\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [20]:
## lets convert df_encoded in boolean type again as i am getting Deprecation warning because support of non-bool type might get discontinued in future..
df_encoded = df_encoded.astype(bool)

In [21]:
frequent_itemsets = apriori(df_encoded, min_support = 0.005, use_colnames = True)
frequent_itemsets.head(2)

,support,itemsets
0,0.020267,(almonds)
1,0.008800,(antioxydant juice)


In [22]:
## Generate association rules..
from mlxtend.frequent_patterns import association_rules
rules = association_rules(frequent_itemsets, metric = "lift", min_threshold = 1)

In [23]:
rules.head(2)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(almonds),(burgers),0.020267,0.087200,0.0052,0.256579,2.942419,1.0,0.003433,1.227837,0.673799,0.050847,0.185560,0.158106
1,(burgers),(almonds),0.087200,0.020267,0.0052,0.059633,2.942419,1.0,0.003433,1.041863,0.723207,0.050847,0.040181,0.158106


In [24]:
rules = rules[(rules['confidence'] >= 0.2) & (rules['lift'] > 1)]

In [25]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(almonds),(burgers),0.020267,0.087200,0.005200,0.256579,2.942419,1.0,0.003433,1.227837,0.673799,0.050847,0.185560,0.158106
2,(almonds),(chocolate),0.020267,0.163867,0.006000,0.296053,1.806668,1.0,0.002679,1.187778,0.455731,0.033683,0.158092,0.166334
4,(almonds),(eggs),0.020267,0.179733,0.006533,0.322368,1.793593,1.0,0.002891,1.210491,0.451613,0.033770,0.173889,0.179359
6,(almonds),(milk),0.020267,0.129600,0.005200,0.256579,1.979776,1.0,0.002573,1.170804,0.505130,0.035945,0.145886,0.148351
8,(almonds),(mineral water),0.020267,0.238267,0.007467,0.368421,1.546255,1.0,0.002638,1.206078,0.360584,0.029740,0.170866,0.199879


In [26]:
## Earliar i chose minimum support as 0.02.. it found out to be too strict hence i wasn't catching any meaningful patters (no frequent itemset)..
## Then lowered the minimum support to 0.005 which is working..
## This indicates that the dataset contains sparce item combinations..
## Hence a lower minimum support is required to catch meaningful associations..
## High support --> fewer but strong patterns
## low support --> more patterns but may include weaker ones..

In [27]:
## Lets analysis top 5 rules from rules.head()

## Almonds --> Burgers (Strongest Rule).. Confidence~25.6%, Lift~2.94 (very high)..
## Customer buying almonds are 3 times more likely to by burgers..very strong and meaningful relationship..

## Similarly.. Almonds --> Milk.. Confidence~25.6%, Lift~1.97.. 
## Almonds buyer also tend to buy milk .. its kind of logical pairing health and breakfast wise..

## Almonds --> Chocolate.. Confidence~29.6%, Lift~1.80..
## customers buying almonds also buy chocolate frequently.. its suggests kind of snacking behavior of customer..

## Almonds --> Eggs.. Confidence~32.2% , Lift~1.79..
## Almonds buyers also buys eggs regularly..it somehow suggests nutrition focused customers..

## Almonds --> Mineral water..Confidence~36.8%, Lift~1.54..
## Strong tendency to buy drinks along with almonds.. Good for bundle offers (snack + drinks)..

## Overall almonds appears to be a common trigger item..almond buyers have diverse purchace behavior. 

In [28]:
## From the above analysis of top 5 items of rules, we can see that almonds act as a central product influencing the purchase of various products..
## Product such as burgers, milk, chocolate, eggs and mineral water are influenced by almonds..
## This indicates that customers purchasing almonds tend to have diverse buying patterns suggesting opportunities for cross selling and product bundling..
## Even though the support is low, high lift value indicates strong associations between items..

In [29]:
rules.shape

(587, 14)

### Analysis and Interpretation:

In [30]:
## •	Analyse the generated rules to identify interesting patterns and relationships between the products.
## •	Interpret the results and provide insights into customer purchasing behaviour based on the discovered rules.

In [31]:
## Since the total numeber of generated rules is large (around 587), it is not practical to interpret each of them.. 
## So lets select a subset of the most relevant rules and analyze them..
## Lets perform exploratory checks on the generated rules to identify the most frequently occuring item combinations (antecedents)..
## Also one to highlight the stongest associations based on the lift metric..

In [32]:
rules['antecedents'].value_counts().head()

antecedents
(frozen vegetables, spaghetti)    8
(spaghetti, olive oil)            7
(french fries, spaghetti)         7
(chocolate, frozen vegetables)    7
(almonds)                         6
Name: count, dtype: int64

In [33]:
rules.sort_values(by = 'lift', ascending = False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
375,(pasta),(escalope),0.015733,0.079333,0.005867,0.372881,4.700185,1.0,0.004618,1.468090,0.799826,0.065770,0.318843,0.223415
764,(pasta),(shrimp),0.015733,0.071333,0.005067,0.322034,4.514494,1.0,0.003944,1.369783,0.790935,0.061789,0.269958,0.196531
749,(whole wheat pasta),(olive oil),0.029467,0.065733,0.008000,0.271493,4.130221,1.0,0.006063,1.282441,0.780893,0.091743,0.220237,0.196599
1616,"(herb & pepper, spaghetti)",(ground beef),0.016267,0.098267,0.006400,0.393443,4.003826,1.0,0.004802,1.486641,0.762645,0.059186,0.327343,0.229286
1610,"(mineral water, herb & pepper)",(ground beef),0.017067,0.098267,0.006667,0.390625,3.975153,1.0,0.004990,1.479768,0.761432,0.061350,0.324218,0.229234


In [34]:
## The strongest product Associations..
## Pasta --> Escalope (Lift ~ 4.7), Pasta --> Shrimp (Lift ~ 4.5), Whole wheat pasta --> Olive oil (Lift ~ 4.13) 
## Spaghetti + Herbs + pepper --> Ground beef (Lift ~ 4.0), Herbs + pepper + Milneral water --> Ground beef (Lift ~ 3.9)
## Above mentioned combinations have very strong relationships.. Customers buying these items are much more likely to buy the paired items..
## These are meal based combinations.. customers are not randomly buying, they are buying ingredients for complete meals..

In [35]:
## Frequent item combinations (from antecedents count)..
## (spaghetti, frozen vegetables), (french fries, spaghetti), (frozen vegetables, chocolate), (olive oil, spaghetti)..
## From this we can say that Spaghetti is a key driver product, meaning it has high cross selling potential..

In [36]:
## Confidence analysis..
## Pasta --> Escalope (Confidence ~ 0.37), Spaghetti combo --> Ground beef (Confidence ~ 0.39)
## They have around 30-40 percent probability.. its not extremely high but combined with high lift makes it meaning full..
## Even moderate confidence is valuable when lift is high.. this situiation can surely be considered as true association, not random..

In [37]:
## Support Values..
## Support (0.005 to 0.008) are quite low.. these combinations are not frequent overall but still quite important as the retail dataset is large..
## These are niche but valuable patterns not mass behavior.. 

In [38]:
## Customer behaviour..
## From patterns that we have observed we can say that customers buy complimentary items more.. (Pasta + sause/meat), (Spaghetti + vegetables/meat)..
## Shopping is goal oriented, means kind os planned meal combinations according to goals..
## Certain product act as an anchors like Spaghetti, Pasta, Olive oil etc..

In [39]:
## What business insight that we acquired from above rules..Place related items together like Pasta near sauces/meat..
## Provide bundle offers like buy Pasta + Shrimp and get discounts..Promote anchor products like Spaghetti based combos..
## Recommend items to customers like, "Customers who bought this also bought.."

In [40]:
## Conclusion..
## The associoation rule analysis reveals strong relationships between complementary food items, 
## It indicates that customers tend to purchase products in combinations based on meal prepration, with certain items like pasta and spaghetti acting as key drivers for cross selling opportunities.. 

### Interview Questions:

In [41]:
## • 1.	What is lift and why is it important in Association rules?
## • 2.	What is support and Confidence. How do you calculate them?
## • 3.	What are some limitations or challenges of Association rules mining?

In [42]:
## Lift measures how much more likely two items occur together compared to if they were independent..
## [Lift(AB) = Confidence(AB)/Support(B)]
## If Lift = 1 means No relationship, independent..If Lift > 1 means Positive association, rule is important..If Lift < 1 means negative association..
## Confidence alone can be misleading because some items are very frequent..
## Lift tells the real strength of the relatioship between items..It helps identify meaningful product combinations..

In [43]:
## Support measures how frequently an itemset appears in the datset..
## [Support(A) = Transaction containing A/Total Transactions]
## [Support(AB) = Transaction containing A and B/ Total Transactions]

## Confidence measures how often B appears when A is present..
## [Confidence(AB) = Support(A INTERSECTION b)/Support(A)]

## How do we calculate them? Lets take an small example..
## Lets say: 100 total transactions, 20 contain milk, 10 contain milk and bread..
## then Support(milk-->bread) = 10/100 = 0.10
## Confidence(milk-->bread) = 10/20 = 0.50

In [44]:
## Limitations or challenges of Association rules mining
## 1st limitation that i felt was too many rules generated, like in my case, around 587 which are difficult to interpret.. might miss out meaningful onces..
## Threshold tuning has to be done with care.. high support might miss usefull patterns and low support will result too many weak rules..
## We cant blindly beleive the rules, just because 2 items were brought together that doesn't mean that they are powerfull rule.. 
## there could be some chances of coincidence as well in case of some transactions..
## Some important items might miss out the rules just because they are not frequently bought.. for example maybe costly items..
## It only looks at co-occurance of items together, not order of purchase.. it does not consider sequence or time..
## This algorithm can be computationally expensive, especially with large datasets and a high number of unique items..